# 成员 2：Task 1 技术面择时指标

**我的任务**：读 1 号同学交付的清洗面板，实现 MA / 动量 / 量价三类信号 + RSI/MACD/布林/波动率等进阶指标，
做 0/1 信号与 expanding 百分位两种打分，合成技术面得分（tech_score）。

**输入**：`数据/step1_panel.csv`（1 号产出，已贴进本文件夹）
**输出**：`输出/step2_panel.csv`（在原面板上追加全部技术指标与得分列，交给 3 号）
**交接**：向 3 号讲清楚——三类信号为什么等权合成；expanding 百分位为什么不是全样本 rank（防前视）。

In [1]:
from __future__ import annotations
import io, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from types import SimpleNamespace

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

BASE = os.getcwd()
RAW = os.path.join(BASE, '数据')
OUT = os.path.join(BASE, '输出')
os.makedirs(OUT, exist_ok=True)
STOCK_NAME = '牧原股份'
STOCK_CODE = '002714.SZ'
TRADING_DAYS = 243


In [2]:
class TechnicalFramework(object):
    """
    题目 Table2 要求三类至少各 1 个指标，这里每类做 2-3 个，另加 RSI/MACD/BOLL/波动率等进阶指标。
    评分口径按题目 Task1 第 4 点两种都做：
      (a) 0/1 信号本身 -> 0 或 100 分
      (b) 连续型指标   -> 在该股自己历史上的 expanding 百分位（0-100）
    所有指标一律用后复权价 close_adj 计算，避免除权缺口把 moving average 打断。
    """

    def __init__(self, panel: pd.DataFrame, min_periods=60):
        self.px = panel
        self.min_p = min_periods

    # ---- 3.1 扩展窗口百分位（只用 t 及之前的信息，杜绝前视偏差）----
    def expanding_pctile(self, s, direction=1):
        x = pd.Series(s, dtype="float64")
        try:
            p = x.expanding(min_periods=self.min_p).rank(pct=True) * 100
        except AttributeError:                     # 老版本 pandas 没有 expanding.rank
            p = x.expanding(min_periods=self.min_p).apply(
                lambda w: (w <= w.iloc[-1]).mean() * 100, raw=False)
        if direction < 0:                          # 越小越好的指标要翻转
            p = 100 - p
        return p

    # ---- 3.2 均线族 ----
    def add_ma_rules(self):
        px = self.px
        for w in [5, 10, 20, 30, 60, 120, 250]:
            px["ma%d" % w] = px["close_adj"].rolling(w).mean()
        px["vwma20"] = (px["close_adj"] * px["volume"]).rolling(20).sum() / px["volume"].rolling(20).sum()
        self.ma_pairs = [(5, 20), (10, 60), (20, 120), (60, 250)]
        for s, l in self.ma_pairs:
            px["sig_ma_%d_%d" % (s, l)] = (px["ma%d" % s] >= px["ma%d" % l]).astype(float)
        # 均线缺口（连续型）：短均线相对长均线的偏离幅度
        px["ma_gap_20_60"] = px["ma20"] / px["ma60"] - 1.0
        px["score_ma_gap"] = self.expanding_pctile(px["ma_gap_20_60"])
        return px

    # ---- 3.3 动量族 ----
    def add_momentum_rules(self):
        px = self.px
        self.mom_windows = [20, 60, 120, 250]
        for n in self.mom_windows:
            px["mom%d" % n] = px["close_adj"] / px["close_adj"].shift(n) - 1.0
            px["sig_mom_%d" % n] = (px["mom%d" % n] >= 0).astype(float)
            px["score_mom_%d" % n] = self.expanding_pctile(px["mom%d" % n])
        return px

    # ---- 3.4 量价族（VWMA 规则 + 量能）----
    def add_price_volume_rules(self):
        px = self.px
        self.vwma_pairs = [(5, 30), (10, 60)]
        for s, l in self.vwma_pairs:
            vs = (px["close_adj"] * px["volume"]).rolling(s).sum() / px["volume"].rolling(s).sum()
            vl = (px["close_adj"] * px["volume"]).rolling(l).sum() / px["volume"].rolling(l).sum()
            px["vwma%d" % s] = vs
            px["vwma%d" % l] = vl
            px["sig_vwma_%d_%d" % (s, l)] = (vs >= vl).astype(float)
        px["vol_ratio_20_60"] = px["volume"].rolling(20).mean() / px["volume"].rolling(60).mean()
        px["sig_vol_expand"] = (px["vol_ratio_20_60"] >= 1).astype(float)
        px["turnover_ma20"] = px["turnover"].rolling(20).mean()
        px["score_turnover"] = self.expanding_pctile(px["turnover_ma20"])
        return px

    # ---- 3.5 进阶指标（加分项）----
    def add_advanced(self):
        px = self.px
        # RSI(14)：Wilder 平滑，用 ewm(alpha=1/14) 实现
        chg = px["close_adj"].diff()
        gain = chg.clip(lower=0)
        loss = (-chg).clip(lower=0)
        avg_gain = gain.ewm(alpha=1 / 14.0, min_periods=14, adjust=False).mean()
        avg_loss = loss.ewm(alpha=1 / 14.0, min_periods=14, adjust=False).mean()
        rs = avg_gain / avg_loss.replace(0, np.nan)
        px["rsi14"] = 100 - 100 / (1 + rs)
        px["sig_rsi_50"] = (px["rsi14"] >= 50).astype(float)
        px["score_rsi"] = self.expanding_pctile(px["rsi14"])

        # MACD(12,26,9)
        ema12 = px["close_adj"].ewm(span=12, adjust=False).mean()
        ema26 = px["close_adj"].ewm(span=26, adjust=False).mean()
        px["macd_dif"] = ema12 - ema26
        px["macd_dea"] = px["macd_dif"].ewm(span=9, adjust=False).mean()
        px["macd_hist"] = px["macd_dif"] - px["macd_dea"]
        px["sig_macd"] = (px["macd_dif"] >= px["macd_dea"]).astype(float)
        px["score_macd"] = self.expanding_pctile(px["macd_hist"])

        # 布林带 %B(20,2)
        mid = px["ma20"]
        sd = px["close_adj"].rolling(20).std()
        px["boll_upper"] = mid + 2 * sd
        px["boll_lower"] = mid - 2 * sd
        px["boll_pctB"] = (px["close_adj"] - px["boll_lower"]) / (px["boll_upper"] - px["boll_lower"])
        px["sig_boll_mid"] = (px["close_adj"] >= mid).astype(float)

        # 乖离率 BIAS20 与 20 日年化波动率
        px["bias20"] = px["close_adj"] / mid - 1.0
        px["score_bias20"] = self.expanding_pctile(px["bias20"])
        px["volatility20"] = px["ret"].rolling(20).std() * np.sqrt(TRADING_DAYS)
        # 高波动期通常对应猪周期的猪价剧烈波动，作为风险信号：低波动更偏多 -> direction=-1
        px["score_vol_low"] = self.expanding_pctile(px["volatility20"], direction=-1)

        # 相对沪深300 的 20 日超额收益（动量的相对强弱版本）
        px["excess_ret20"] = (px["close_adj"].pct_change(20)
                              - px["idx_close"].pct_change(20))
        px["score_excess20"] = self.expanding_pctile(px["excess_ret20"])
        return px

    # ---- 3.6 技术面综合评分 ----
    def build_score(self):
        px = self.px
        sig_cols = (["sig_ma_%d_%d" % p for p in self.ma_pairs]
                    + ["sig_mom_%d" % n for n in self.mom_windows]
                    + ["sig_vwma_%d_%d" % p for p in self.vwma_pairs]
                    + ["sig_vol_expand", "sig_rsi_50", "sig_macd", "sig_boll_mid"])
        self.signal_cols = sig_cols
        px["tech_signal_mean"] = px[sig_cols].mean(axis=1) * 100   # 0/1 信号的平均胜率式打分
        # 连续型指标打分（题目 Task1 第4点 b 类）
        cont_cols = ["score_ma_gap", "score_mom_20", "score_mom_60", "score_mom_120",
                     "score_rsi", "score_macd", "score_bias20", "score_vol_low", "score_excess20"]
        self.continuous_cols = cont_cols
        px["tech_cont_mean"] = px[cont_cols].mean(axis=1)
        # 技术面择时得分：三类等权（均线类 / 动量类 / 量价类）再各占 1/3，进阶指标不单独加权，
        # 只作为交叉验证；这样与题目 Task1「三类各至少一个」的结构保持一致。
        ma_part = px[["sig_ma_%d_%d" % p for p in self.ma_pairs]].mean(axis=1) * 100
        mom_part = px[["sig_mom_%d" % n for n in self.mom_windows]].mean(axis=1) * 100
        pv_part = px[["sig_vwma_%d_%d" % p for p in self.vwma_pairs] + ["sig_vol_expand"]].mean(axis=1) * 100
        px["score_ma_part"] = ma_part
        px["score_mom_part"] = mom_part
        px["score_pv_part"] = pv_part
        px["tech_score"] = (ma_part + mom_part + pv_part) / 3.0
        px["tech_score_full"] = 0.75 * px["tech_score"] + 0.25 * px[cont_cols].mean(axis=1)
        return px

    # ---- 3.7 描述性统计 + 画图 ----
    def run(self):
        self.add_ma_rules()
        self.add_momentum_rules()
        self.add_price_volume_rules()
        self.add_advanced()
        self.build_score()
        desc_cols = (["tech_score", "tech_signal_mean", "ma_gap_20_60", "mom20", "mom60",
                      "rsi14", "macd_hist", "boll_pctB", "volatility20", "vol_ratio_20_60",
                      "turnover_ma20", "excess_ret20", "ret"])
        stats = self.px[desc_cols].describe(percentiles=[0.2, 0.5, 0.75]).T
        stats = stats[["count", "mean", "std", "min", "20%", "50%", "75%", "max"]]
        stats.to_csv(os.path.join(OUT, "task1_summary_stats.csv"), encoding="utf-8-sig")
        sig_rate = self.px[self.signal_cols].mean().rename("信号为1的比例").to_frame()
        sig_rate.to_csv(os.path.join(OUT, "task1_signal_hit_rate.csv"), encoding="utf-8-sig")
        self.plot()
        return self.px, stats, sig_rate

    def plot(self):
        px = self.px
        fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True,
                                 gridspec_kw={"height_ratios": [3, 1, 1, 1]})
        ax = axes[0]
        ax.plot(px.index, px["close_adj"], lw=1.0, color="#222222", label="后复权收盘价")
        for w, c in [(20, "#1f77b4"), (60, "#ff7f0e"), (120, "#2ca02c")]:
            ax.plot(px.index, px["ma%d" % w], lw=0.9, color=c, label="MA%d" % w, alpha=0.85)
        # 信号阴影：均线多头排列期
        bull = (px["sig_ma_20_120"] == 1) & (px["sig_mom_60"] == 1)
        ax.fill_between(px.index, px["close_adj"].min() * 0.9, px["close_adj"].max() * 1.05,
                        where=bull, color="#2ca02c", alpha=0.10, step="pre",
                        label="多头排列区间(MA20>MA120 且 60日动量为正)")
        ax.set_yscale("log")
        ax.set_title("%s(%s) 后复权价格与均线/信号（2014 上市至今）" % (STOCK_NAME, STOCK_CODE))
        ax.legend(loc="upper left", fontsize=8, ncol=2)
        ax.grid(alpha=0.25)

        axes[1].plot(px.index, px["tech_score"], color="#1f77b4", lw=0.8)
        axes[1].axhline(50, color="gray", lw=0.7, ls="--")
        axes[1].set_ylabel("技术面得分")
        axes[1].set_ylim(-2, 102)
        axes[1].grid(alpha=0.25)

        axes[2].plot(px.index, px["rsi14"], color="#9467bd", lw=0.7, label="RSI14")
        axes[2].axhline(50, color="gray", lw=0.6, ls="--")
        axes[2].set_ylabel("RSI")
        axes[2].legend(fontsize=8, loc="upper left")
        axes[2].grid(alpha=0.25)

        axes[3].plot(px.index, px["vol_ratio_20_60"], color="#8c564b", lw=0.7, label="20/60日均量比")
        axes[3].axhline(1, color="gray", lw=0.6, ls="--")
        axes[3].set_ylabel("量比")
        axes[3].legend(fontsize=8, loc="upper left")
        axes[3].grid(alpha=0.25)
        axes[3].xaxis.set_major_locator(mdates.YearLocator())
        axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        fig.tight_layout()
        p = os.path.join(OUT, "task1_technical.png")
        fig.savefig(p, dpi=140)
        plt.close(fig)
        print("  [图] 已保存 %s" % p)


# ====================================================================================
# SECTION 4: Task2 基本面择时指标（六大维度 + Point-in-Time 对齐）
# ====================================================================================

In [3]:
# ---- 读 1 号的清洗面板 ----
panel = pd.read_csv(os.path.join(RAW, 'step1_panel.csv'), parse_dates=['trade_date'])
panel = panel.set_index('trade_date').sort_index()
print('读入面板', panel.shape, panel.index.min().date(), '~', panel.index.max().date())

tech = TechnicalFramework(panel)
panel, stats, sig_rate = tech.run()
panel.to_csv(os.path.join(OUT, 'step2_panel.csv'), encoding='utf-8-sig')
print('step2 完成：面板', panel.shape, '技术面得分列 tech_score 已加入')
print(stats.round(4).to_string())

读入面板 (3057, 29) 2014-01-28 ~ 2026-09-30


  [图] 已保存 C:\Users\35632\Desktop\python期中大作业\小组版\2\我的代码\输出\task1_technical.png
step2 完成：面板 (3057, 90) 技术面得分列 tech_score 已加入
                   count     mean      std      min      20%      50%      75%       max
tech_score        3057.0  54.2834  30.5510   0.0000  25.0000  55.5556  80.5556  100.0000
tech_signal_mean  3057.0  54.1287  29.8225   0.0000  21.4286  57.1429  78.5714  100.0000
ma_gap_20_60      2998.0   0.0219   0.0822  -0.1778  -0.0410   0.0084   0.0718    0.3235
mom20             3037.0   0.0292   0.1324  -0.3772  -0.0687   0.0138   0.0923    0.8839
mom60             2997.0   0.0905   0.2537  -0.4926  -0.0992   0.0361   0.2071    1.4547
rsi14             3043.0  51.5931  12.2161  15.1529  41.3814  51.4086  59.7108   89.0220
macd_hist         3057.0   0.0107   6.3506 -38.1835  -3.1268   0.0606   2.0560   40.6088
boll_pctB         3038.0   0.5266   0.3253  -0.4036   0.2142   0.5334   0.7879    1.3836
volatility20      3037.0   0.4060   0.1896   0.0975   0.2418   0.3807   0.5